# Global Space Flight Analytics & Vehicle Reliability (1957–2020)
### Exploratory Data Analysis & Strategic Aerospace Trend Modeling

This notebook provides the core data preparation, exploratory analysis, and visualization pipeline for historical orbital launches from 1957 to 2020.

### 1. Environment Setup & Core Dependencies

In [ ]:
%pip install iso3166
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import calendar
from iso3166 import countries

pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

### 2. Data Cleaning & Feature Engineering Pipeline

In [ ]:
df_data = pd.read_csv('mission_launches.csv')

# Clean temporal features
df_data['Date_Clean'] = pd.to_datetime(df_data['Date'], utc=True, errors='coerce')
df_data['Year'] = df_data['Date_Clean'].dt.year
df_data['Month_Num'] = df_data['Date_Clean'].dt.month
df_data['Month_Name'] = df_data['Date_Clean'].dt.strftime('%B')
df_data['Decade'] = (df_data['Year'] // 10) * 10
df_data['Decade_Label'] = df_data['Decade'].astype(str) + 's'

# Binary reliability flags
df_data['Is_Success'] = (df_data['Mission_Status'].astype(str).str.strip() == 'Success').astype(int)
df_data['Is_Failure'] = (df_data['Mission_Status'].astype(str).str.strip().isin([
    'Failure', 'Partial Failure', 'Prelaunch Failure'
])).astype(int)

# Geographic ISO alpha-3 extraction
COUNTRY_OVERRIDES = {
    'Russia': 'Russian Federation',
    'New Mexico': 'United States of America',
    'USA': 'United States of America',
    'UK': 'United Kingdom of Great Britain and Northern Ireland',
    'South Korea': 'Korea, Republic of',
    'North Korea': "Korea, Democratic People's Republic of",
    'Iran': 'Iran, Islamic Republic of',
    'Vietnam': 'Viet Nam',
    'Syria': 'Syrian Arab Republic',
    'Kazakhstan': 'Kazakhstan',
    'Barents Sea': 'Russian Federation',
    'Yellow Sea': 'China',
    'Pacific Ocean': 'United States of America',
    'Shahrud Missile Test Site': 'Iran, Islamic Republic of',
    'Pacific Missile Range Facility': 'United States of America',
    'Gran Canaria': 'Spain'
}

def get_alpha3(country_name):
    if pd.isna(country_name):
        return None
    c_clean = str(country_name).strip()
    c_std = COUNTRY_OVERRIDES.get(c_clean, c_clean)
    try:
        return countries.get(c_std).alpha3
    except (KeyError, AttributeError):
        return None

df_data['Country_Name'] = df_data['Location'].apply(lambda x: x.split(',')[-1].strip())
df_data['ISO_Alpha3'] = df_data['Country_Name'].apply(get_alpha3)

# Geopolitical Bloc Attribution
def map_bloc(row):
    iso = str(row.get('ISO_Alpha3', '')).strip().upper()
    c_name = str(row.get('Country_Name', '')).strip().upper()
    if iso in ['RUS', 'KAZ'] or c_name in ['RUSSIA', 'KAZAKHSTAN', 'USSR']:
        return 'USSR / Russia'
    if iso == 'USA' or c_name in ['USA', 'UNITED STATES']:
        return 'USA'
    if iso == 'CHN' or c_name in ['CHINA']:
        return 'China'
    if iso == 'FRA' or c_name in ['FRANCE']:
        return 'Europe / France'
    if iso == 'IND' or c_name in ['INDIA']:
        return 'India'
    if iso == 'JPN' or c_name in ['JAPAN']:
        return 'Japan'
    return 'Rest of World'

df_data['Bloc'] = df_data.apply(map_bloc, axis=1)

# Numeric price conversion
df_data['Price_Numeric'] = pd.to_numeric(
    df_data['Price'].astype(str).str.replace(',', '', regex=False).str.strip(),
    errors='coerce'
)

### 3. Launch Cost Distribution & Tiered Reliability Modeling

In [ ]:
df_priced = df_data[df_data['Price_Numeric'] > 0].copy()
bins = [0, 20, 100, np.inf]
labels = [
    'Tier 1: Small-Sat / Budget (< $20M)',
    'Tier 2: Medium/Commercial Workhorses ($20M–$100M)',
    'Tier 3: Heavy Lifters & Human Flight (> $100M)'
]
df_priced['Cost_Tier'] = pd.cut(df_priced['Price_Numeric'], bins=bins, labels=labels, right=True)

tier_summary = (
    df_priced.groupby('Cost_Tier', observed=True)
    .agg(
        Total_Missions=('Is_Success', 'count'),
        Successful_Missions=('Is_Success', 'sum'),
        Median_Price=('Price_Numeric', 'median'),
        Mean_Price=('Price_Numeric', 'mean')
    )
    .reset_index()
)
tier_summary['Success_Rate_%'] = (tier_summary['Successful_Missions'] / tier_summary['Total_Missions']) * 100
display(tier_summary)

# Log-Scale Cost Distribution Plot
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(
    data=df_priced,
    x='Price_Numeric',
    log_scale=True,
    bins=25,
    color='#2563eb',
    edgecolor='#0f172a',
    ax=ax
)
ticks = [5, 10, 25, 50, 100, 250, 500, 1000, 5000]
ax.set_xticks(ticks)
ax.set_xticklabels(['$5M', '$10M', '$25M', '$50M', '$100M', '$250M', '$500M', '$1.0B', '$5.0B'])
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_title('Distribution of Disclosed Launch Costs (Log Scale)', fontsize=13, pad=12)
ax.set_xlabel('Launch Price (Millions USD)', fontsize=11)
ax.set_ylabel('Number of Missions', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 4. Failure Dynamics & Engineering Learning Curve (Year-on-Year)

In [ ]:
annual_failures = (
    df_data[df_data['Year'].notna() & (df_data['Year'] <= 2020)]
    .groupby('Year')
    .agg(
        Total_Launches=('Mission_Status', 'count'),
        Failed_Launches=('Is_Failure', 'sum')
    )
    .reset_index()
)
annual_failures['Year'] = annual_failures['Year'].astype(int)
annual_failures['Failure_Rate_%'] = (annual_failures['Failed_Launches'] / annual_failures['Total_Launches']) * 100

# Dual-Axis Failure Volume vs. Failure Rate
fig_fail = make_subplots(specs=[[{"secondary_y": True}]])
fig_fail.add_trace(
    go.Bar(
        x=annual_failures['Year'],
        y=annual_failures['Failed_Launches'],
        name='Failed Launches (Count)',
        marker_color='#ef4444',
        opacity=0.6
    ),
    secondary_y=False
)
fig_fail.add_trace(
    go.Scatter(
        x=annual_failures['Year'],
        y=annual_failures['Failure_Rate_%'],
        name='Failure Rate (%)',
        mode='lines+markers',
        line=dict(color='#0f172a', width=2.5),
        marker=dict(size=4)
    ),
    secondary_y=True
)
fig_fail.update_layout(
    title='Orbital Mission Failures & Industry Failure Rate (1957–2020)',
    width=1100,
    height=550,
    hovermode='x unified',
    plot_bgcolor='white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.85)')
)
fig_fail.update_yaxes(title_text='Failed Launches (Count)', secondary_y=False, showgrid=True, gridcolor='#f1f5f9')
fig_fail.update_yaxes(title_text='Failure Rate (%)', secondary_y=True, showgrid=False, ticksuffix='%', range=[0, 70])
fig_fail.update_xaxes(title_text='Year', dtick=5, showgrid=True, gridcolor='#f1f5f9')
fig_fail.show()

### 5. Multipolar Market Share & Decade Reliability Matrix

In [ ]:
# Annual Launches by Geopolitical Bloc
annual_blocs = (
    df_data[df_data['Year'].notna() & (df_data['Year'] <= 2020)]
    .groupby(['Year', 'Bloc'])
    .size()
    .reset_index(name='Launches')
)
fig_share = px.bar(
    annual_blocs,
    x='Year',
    y='Launches',
    color='Bloc',
    title='Global Annual Launch Distribution by Space Power (1957–2020)',
    color_discrete_map={
        'USA': '#2563eb',
        'USSR / Russia': '#dc2626',
        'China': '#f59e0b',
        'Europe / France': '#10b981',
        'India': '#8b5cf6',
        'Japan': '#ec4899',
        'Rest of World': '#94a3b8'
    }
)
fig_share.update_layout(barmode='stack', width=1100, height=550, plot_bgcolor='white')
fig_share.update_xaxes(showgrid=True, gridcolor='#f1f5f9', dtick=5)
fig_share.update_yaxes(showgrid=True, gridcolor='#f1f5f9')
fig_share.show()

# Decade x Bloc Reliability Matrix
decade_stats = (
    df_data[df_data['Year'].notna() & (df_data['Year'] <= 2020)]
    .groupby(['Decade_Label', 'Bloc'])
    .agg(Total=('Mission_Status', 'count'), Successes=('Is_Success', 'sum'))
    .reset_index()
)
decade_stats['Success_Rate_%'] = (decade_stats['Successes'] / decade_stats['Total']) * 100
heatmap_matrix = decade_stats.pivot(index='Bloc', columns='Decade_Label', values='Success_Rate_%')
decades_ordered = [f"{d}s" for d in range(1950, 2030, 10) if f"{d}s" in heatmap_matrix.columns]
heatmap_matrix = heatmap_matrix[decades_ordered]

fig_hm = px.imshow(
    heatmap_matrix,
    text_auto='.1f',
    aspect='auto',
    color_continuous_scale='RdYlGn',
    range_color=[70, 100],
    title='Launch Success Rate (%) by Space Power Across Decades',
    labels=dict(x="Decade", y="Space Power", color="Success Rate (%)")
)
fig_hm.update_layout(width=950, height=450)
fig_hm.show()

### 6. Clean Analytical Export for Tableau Dashboard

In [ ]:
# Export clean schema matching the Tableau workbook requirements
export_cols = [
    'Detail', 'Organisation', 'Location', 'Country_Name', 'ISO_Alpha3', 'Bloc',
    'Date_Clean', 'Year', 'Decade_Label', 'Rocket_Status', 'Price_Numeric',
    'Mission_Status', 'Is_Success', 'Is_Failure'
]
df_export = df_data.loc[df_data['Year'].notna() & (df_data['Year'] <= 2020), export_cols].copy()
df_export.rename(columns={'Price_Numeric': 'Price_USD_M'}, inplace=True)
df_export.to_csv('space_launches_tableau.csv', index=False)
print(f"Successfully exported 'space_launches_tableau.csv' with {len(df_export)} records.")